Step1: Import libraries

In [0]:
import requests
import pandas as pd
import numpy as np
import json 
import psycopg2
import datetime as dt
import time

# Set the option to display all columns
#pd.set_option('display.max_columns',100)

# Bronze Layer 

- API connect 
- Read Jason file 
- Parse & Reshape data as needed

In [0]:
def fetch_ibm_daily_data(api_key='demo'):
    """
    Fetches daily IBM stock data from Alpha Vantage API and returns a formatted Pandas DataFrame.

    - Calls the TIME_SERIES_DAILY endpoint for IBM.
    - Extracts date, open, high, low, close, and volume for each day.
    - Adds a 'Last_updated' column from API metadata.
    - Returns a DataFrame with columns: Date, Open, High, Low, Close, Volume, Last_updated.
    """
    url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey={api_key}'
    response = requests.get(url).json()
    data = response['Time Series (Daily)']
    dates = pd.DataFrame(data.keys())
    info = pd.DataFrame(data.values())
    last_updated = pd.DataFrame([response['Meta Data']['3. Last Refreshed']], columns=['Last_updated'])
    result = pd.concat([dates, info, last_updated], axis=1)
    result['Last_updated'] = result['Last_updated'].fillna(method='ffill')
    result = result.rename(columns={0: 'Date', '1. open': 'Open', '2. high': 'High', '3. low': 'Low', '4. close': 'Close', '5. volume': 'Volume'})
    return result

result = fetch_ibm_daily_data()

In [0]:
spark_df = spark.createDataFrame(result)


### To Check if bronze table exists
table_exists = spark.catalog.tableExists("workspace.bronze.ibm")
table_exists

In [0]:
# Compare the schema (column names and types) of the local Spark DataFrame and the bronze.ibm table for validation
spark_df_schema = spark_df.dtypes

bronze_ibm_schema = spark.table("workspace.bronze.ibm").dtypes

display(pd.DataFrame({
    "spark_df": dict(spark_df_schema),
    "bronze_ibm": dict(bronze_ibm_schema)
}))

In [0]:
# Cast columns to appropriate types for Spark DataFrame and display schema
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, LongType, DateType

spark_df = spark.createDataFrame(result)
spark_df = spark_df.withColumn("Date", col("Date").cast(DateType())) \
                   .withColumn("Open", col("Open").cast(DoubleType())) \
                   .withColumn("High", col("High").cast(DoubleType())) \
                   .withColumn("Low", col("Low").cast(DoubleType())) \
                   .withColumn("Close", col("Close").cast(DoubleType())) \
                   .withColumn("Volume", col("Volume").cast(LongType())) \
                   .withColumn("Last_updated", col("Last_updated").cast(DateType()))

display(spark_df.describe())

### Append data to bronze.ibm  using timestamp waterMark 
Alterntively we can use : 
`from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "workspace.bronze.ibm")

(
    target.alias("target").merge(
        source=spark_df.alias("source"),
        condition="target.Date = source.Date"
    )
    .whenNotMatchedInsertAll()   # only inserts rows where Date doesn't exist
    .execute()
)`


In [0]:
# Filter only new rows to append to the bronze.ibm table based on the latest Last_updated value
# 1. Get the max Last_updated already in the table
bronze_max_date = spark.sql("SELECT MAX(date) FROM workspace.bronze.ibm").collect()[0][0]

print(f"Max date in table: {bronze_max_date}")

# 2. Filter only new rows from the current DataFrame
new_rows = spark_df.filter(col("date") > bronze_max_date)

print(f"New rows to append: {new_rows.count()}")

#### Append only new rows to the bronze.ibm Delta table

 Filter new rows to append to the bronze.ibm table based on the latest Last_updated value


In [0]:
new_rows.write.format("delta").mode("append").saveAsTable("workspace.bronze.ibm")

Check Delta table history (shows all write operations)


In [0]:
spark.sql("DESCRIBE HISTORY workspace.bronze.ibm").display()


 Check only the rows added in the latest version

In [0]:
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "workspace.bronze.ibm")
latest_version = dt.history(1).select("version").collect()[0][0]

spark.read.format("delta") \
    .option("versionAsOf", latest_version) \
    .table("workspace.bronze.ibm") \
    .display()


# Silver Layer 

Extract - Transfer - Load (Silver.IBM)

### Transformations: 
- Reads the IBM bronze table, converts it to a Pandas DataFrame, 
- groups by 'Date' to get the latest records,
- applies data quality checks (nulls, types, ranges, regex, referential integrity, deduplication),
- and creates or replaces a temporary view named 'silver_ibm'.


grouped_silver is a Spark DataFrame. This is because the result of a groupBy followed by an aggregation (such as .agg({"Date": "max"})) and column renaming (with .withColumnRenamed) on a Spark DataFrame returns another Spark DataFrame

In [0]:
import re
from pyspark.sql.functions import col, lit, regexp_replace, to_date, to_timestamp

# Read the bronze table
df = spark.table("workspace.bronze.ibm").select("Date", "Open", "High", "Low", "Close", "Volume", "Last_updated")



Filter to the max date:
This keeps only the rows where Date equals the most recent date in the DataFrame.




create Delta table as a target 

In [0]:
from delta.tables import DeltaTable

# Step 1: Create silver table if it doesn't exist yet
spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.silver.ibm
    USING DELTA
    AS SELECT * FROM workspace.bronze.ibm WHERE 1=0
""")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "workspace.silver.ibm")
source = spark.sql("SELECT DISTINCT * FROM workspace.bronze.ibm")

target.alias("target").merge(
    source=source.alias("source"),
    condition="target.Date = source.Date"   # ✅ Date is the unique key for stock data
).whenNotMatchedInsertAll() \
 .execute()


### Why use this instead of spark.read.table()?
- python# This gives you a DataFrame (read-only)\
df = spark.read.table("workspace.silver.ibm")

- This gives you a DeltaTable (read + write + Delta ops)\
- target = DeltaTable.forName(spark, "workspace.silver.ibm")


Ulternatively,  Full Silver Cleaning Before Merge: 

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, to_date, to_timestamp
from pyspark.sql.types import DoubleType, LongType

# Step 1: Read bronze
source_raw = spark.sql("SELECT * FROM workspace.bronze.ibm")

# Step 2: Drop nulls on critical columns
source_no_nulls = source_raw.dropna(subset=["Date", "Close", "Open", "High", "Low", "Volume"])

# Step 3: Deduplicate on the merge key
source_deduped = source_no_nulls.dropDuplicates(["Date"])

# Step 4: Cast to correct data types
source_clean = (source_deduped
    .withColumn("Date",         to_date(col("Date"), "yyyy-MM-dd"))
    .withColumn("Open",         col("Open").cast(DoubleType()))
    .withColumn("High",         col("High").cast(DoubleType()))
    .withColumn("Low",          col("Low").cast(DoubleType()))
    .withColumn("Close",        col("Close").cast(DoubleType()))
    .withColumn("Volume",       col("Volume").cast(LongType()))
    .withColumn("Last_updated", to_timestamp(col("Last_updated")))
    .drop("_rescued_data")      # Drop Auto Loader's error capture column
)

# Step 5: Preview before merge — always good practice
print(f"Rows after cleaning: {source_clean.count()}")
source_clean.printSchema()
source_clean.show(5)


In [0]:
print("=== Target (Silver) columns ===")
spark.sql("SELECT * FROM workspace.silver.ibm LIMIT 0").printSchema()

print("=== Source (Cleaned Bronze) columns ===")
source_clean.printSchema()

In [0]:
# Step 6: Merge into silver
target = DeltaTable.forName(spark, "workspace.silver.ibm")

(
    target.alias("target").merge(
        source=source_clean.alias("source"),
        condition="target.Date = source.Date"
    )
    .whenMatchedUpdateAll()     # Update existing rows if Date matches
    .whenNotMatchedInsertAll()  # Insert new rows if Date not found
    .execute()
)

Or We can use SQL to do cleaning into a temperry view then merge like below: 

Analogy
It works the same way as aliasing tables in SQL:
sql-- SQL equivalent
MERGE INTO workspace.silver.ibm AS target
USING workspace.bronze.ibm     AS source   -- this is what source=source_clean.alias("source") does
ON target.Date = source.Date
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *